# 02 — Generators: Functions That `yield`

A **generator** is a function that uses `yield` instead of (or as well as) `return`. Calling it doesn't run the body — it gives you back an *iterator*. Each `next()` runs the body until the next `yield`, then pauses, with all local state intact.

Generators are the easy way to write iterators. You almost never write `__iter__` / `__next__` by hand once you know `yield`.

## A generator vs a function

In [1]:
# Ordinary function — builds a list, returns it.
def squares_list(n):
    out = []
    for i in range(n):
        out.append(i * i)
    return out

# Generator function — yields one square at a time.
def squares_gen(n):
    for i in range(n):
        yield i * i

print(squares_list(5))
print(squares_gen(5))                # generator object — NOT a list yet
print(list(squares_gen(5)))          # materialize it

# Memory: the generator stores ~no state, the list stores all n items.
print(type(squares_gen(5)).__name__)

[0, 1, 4, 9, 16]
<generator object squares_gen at 0x0000019B3E4D2C20>
[0, 1, 4, 9, 16]
generator


## The pause/resume mental model

Each call to `next()` runs the body **until the next `yield`**. Then the function is *paused*: its locals, instruction pointer, even open loops are kept alive. The next call resumes where it left off.

In [2]:
def trace():
    print("  body: start")
    yield "first"
    print("  body: between")
    yield "second"
    print("  body: end")

g = trace()
print("created generator, body NOT yet run")
print("-> first next()");  print("got:", next(g))
print("-> second next()"); print("got:", next(g))
print("-> third next()")
try:
    next(g)
except StopIteration:
    print("  StopIteration as expected")

created generator, body NOT yet run
-> first next()
  body: start
got: first
-> second next()
  body: between
got: second
-> third next()
  body: end
  StopIteration as expected


## `yield from` — delegate to a sub-generator

When your generator wants to yield everything another iterable produces, write `yield from xs` instead of `for x in xs: yield x`. Cleaner, and it handles `StopIteration` / `send` / `throw` correctly.

In [3]:
def chain(*iterables):
    for it in iterables:
        yield from it       # equivalent to: for x in it: yield x

print(list(chain([1, 2], (3, 4), "abc")))   # [1, 2, 3, 4, 'a', 'b', 'c']

[1, 2, 3, 4, 'a', 'b', 'c']


### `yield from` for recursive structures

In [4]:
def flatten(nested):
    """Yield every leaf of an arbitrarily-nested list/tuple."""
    for item in nested:
        if isinstance(item, (list, tuple)):
            yield from flatten(item)     # recurse
        else:
            yield item

print(list(flatten([1, [2, [3, 4], 5], (6, 7), [[[8]]]])))

[1, 2, 3, 4, 5, 6, 7, 8]


## Infinite generators

Because generators are lazy, they can describe infinite sequences. The consumer decides where to stop.

In [5]:
def naturals():
    n = 0
    while True:
        n += 1
        yield n

from itertools import islice
print(list(islice(naturals(), 5)))    # take the first 5

[1, 2, 3, 4, 5]


In [6]:
def fibs():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

# First fib > 1000
for x in fibs():
    if x > 1000:
        print("first fib > 1000:", x)
        break

first fib > 1000: 1597


## Generator expressions — comprehensions with `()`

Same syntax as a list comprehension, just `()`. Builds a generator instead of a list. Memory: O(1) per yield instead of O(n) upfront.

In [7]:
# Heavy: builds a 10-million-int list, sums it, throws it away.
print(sum([i * i for i in range(10_000_000)]))

# Light: never builds the list. Just streams squares into sum().
print(sum(i * i for i in range(10_000_000)))

# When a gen-exp is the SOLE argument to a function, the outer parens are optional:
print(max(x * 2 for x in range(10)))

333333283333335000000


333333283333335000000
18


## The exhausted-twice gotcha

A generator (function or expression) is **one-shot**. After you consume it, it's done — you don't get to iterate again.

In [8]:
xs = (i for i in range(3))
print(list(xs))     # [0, 1, 2]
print(list(xs))     # []  — gotcha

# If you need to re-iterate, materialize to a list or wrap a fresh call:
def fresh():
    return (i for i in range(3))

print(list(fresh()), list(fresh()))

[0, 1, 2]
[]
[0, 1, 2] [0, 1, 2]


## `return` inside a generator

A bare `return` ends the generator (turns into `StopIteration`). `return value` is rare — the value becomes `StopIteration.value`, mostly used in coroutine-style patterns.

In [9]:
def take_until_zero(xs):
    for x in xs:
        if x == 0:
            return        # end the generator early; no more yields
        yield x

print(list(take_until_zero([3, 2, 1, 0, 100, 200])))

[3, 2, 1]


## `close()` / context-manager-like cleanup

Inside a generator, you can use `try / finally` to run cleanup when the generator is closed (consumer stops, generator is garbage-collected, or `g.close()` is called).

In [10]:
def with_cleanup():
    print("  opening resource")
    try:
        for i in range(5):
            yield i
    finally:
        print("  closing resource")

g = with_cleanup()
print(next(g), next(g))
g.close()              # triggers `finally`
# Same thing happens if the consumer `break`s out of a for-loop over g.

  opening resource
0 1
  closing resource


## Mini-exercise

1. Write `evens(n)` that yields the first `n` even non-negative integers. Use it with `sum(evens(100))`.
2. Write `take(n, it)` and `drop(n, it)` — yield/skip the first `n` items of any iterable.
3. Write `pairwise(it)` that yields `(a, b)` for each adjacent pair: `pairwise([1,2,3,4]) -> (1,2), (2,3), (3,4)`.
4. Why does this print `[]`? Fix it without losing laziness.
   ```python
   nums = (n*2 for n in range(5))
   doubled = list(nums)
   tripled = [n*3 for n in nums]   # ?
   print(tripled)
   ```

In [11]:
# your solutions here


**Takeaways**

- `yield` makes a function a generator: each `next()` runs to the next `yield` and pauses.
- `yield from xs` delegates to a sub-iterable; great for recursion and chaining.
- Generator expressions `(x for x in xs)` are lazy comprehensions.
- Generators are **one-shot** — consume once. Re-iterate by calling the function again.
- `try / finally` inside a generator gives you cleanup when the generator is closed.

Next: [03_lazy_pipelines.ipynb](03_lazy_pipelines.ipynb) — compose generators into a real data flow.